In [ ]:
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from datasets import Dataset, Features, load_dataset
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torchsummary import summary

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

cifar = load_dataset("uoft-cs/cifar10")

train_cifar = load_dataset("uoft-cs/cifar10", split="train[:40000]")
val_cifar = load_dataset("uoft-cs/cifar10", split="train[40000:]")
test_cifar = load_dataset("uoft-cs/cifar10", split="test")

In [ ]:
# Convert CIFAR datasets to Torch TensorDataset
def to_tensor_dataset(ds):
    imgs = torch.tensor(np.stack(ds["img"])).float() / 255.0
    labels = torch.tensor(ds["label"]).long()
    reshaped_imgs = imgs.reshape(imgs.shape[0], 3, 32, 32)
    return TensorDataset(reshaped_imgs, labels)


train_dataset = to_tensor_dataset(train_cifar)
val_dataset = to_tensor_dataset(val_cifar)
test_dataset = to_tensor_dataset(test_cifar)

In [ ]:
example_img, example_label = train_dataset.tensors[0][2], train_dataset.tensors[1][2]
img_np = (example_img.numpy() * 255).astype(np.uint8).reshape(32, 32, 3)

plt.imshow(img_np)
plt.title(f"Label: {example_label.item()}")
plt.axis('off')
plt.show()

In [ ]:
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=val_dataset.tensors[0].shape[0], shuffle=False, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=test_dataset.tensors[0].shape[0], shuffle=False, drop_last=True)

print(f"train_data: {train_dataset.tensors[0].dtype}")
print(f"train_labels: {train_dataset.tensors[1].dtype}")
print(f"val_data: {val_dataset.tensors[0].dtype}")
print(f"val_labels: {val_dataset.tensors[1].dtype}")
print(f"test_data: {test_dataset.tensors[0].dtype}")
print(f"test_labels: {test_dataset.tensors[1].dtype}")
print(f"Training data range {train_dataset.tensors[0].min()} - {train_dataset.tensors[0].max()}")
print(f"Val data range {val_dataset.tensors[0].min()} - {val_dataset.tensors[0].max()}")
print(f"Test data range {test_dataset.tensors[0].min()} - {test_dataset.tensors[0].max()}")

In [ ]:
class Cifar10ClassifierModel(nn.Module):

    def __init__(self, printtoggle):
        super(Cifar10ClassifierModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.fc1 = nn.Linear(64 * 8 * 8, 512)
        self.output = nn.Linear(512, 10)

        # toggle for printing out tensor sizes during forward prop
        self.print = printtoggle

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)
        if self.print:
            print(f'Before flattening: {x.shape}')
        x = x.view(x.size(0), -1)  # Flatten the tensor
        if self.print:
            print(f'Vectorize: {x.shape}')
        x = F.relu(self.fc1(x))
        if self.print:
            print(f'fc1: {x.shape}')
        x = self.output(x)
        if self.print:
            print(f'Output: {x.shape}')
        return x

In [ ]:
def create_cifar_net(printtoggle=False, learning_rate=0.001, weight_decay: float = 0.0):
    net = Cifar10ClassifierModel(printtoggle)
    lossfun = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate, weight_decay=weight_decay)

    return net, lossfun, optimizer

In [ ]:
# test the model with one batch
net, loss_func, optimizer = create_cifar_net(True, 0.001, 0.0001)

X, y = next(iter(train_loader))
print(X.shape)
print(y.shape)
y_hat = net(X)

# check sizes of model outputs and target variable
print(' ')
print(y_hat.shape)
print(y.shape)

# now let's compute the loss
loss = loss_func(y_hat, y)
print(' ')
print('Loss:')
print(loss)

In [ ]:
summary(net, input_size=(3, 32, 32), device='cpu')

In [ ]:
def train_cifar_10(num_epochs=10, learning_rate=0.001):
    net, loss_func, optimizer = create_cifar_net(learning_rate=learning_rate)

    losses = torch.zeros(num_epochs, 2)
    train_accuracies = []
    val_accuracies = []

    for epoch_i in tqdm(range(num_epochs), desc="Training"):
        net.train()
        batch_accuracies = []
        batch_loss = []
        for X, y in train_loader:
            # forward pass and loss
            y_hat = net(X)
            loss = loss_func(y_hat, y)

            # backprop
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # loss from this batch
            batch_loss.append(loss.item())

            # compute accuracy
            matches = torch.argmax(y_hat, dim=1) == y  # booleans (false/true)
            matchesNumeric = matches.float()  # convert to numbers (0/1)
            accuracy_pct = 100 * torch.mean(matchesNumeric)  # average and x100
            batch_accuracies.append(accuracy_pct)  # add to list of accuracies
        # end of batch loop...

        # now that we've trained through the batches, get their average training accuracy
        train_accuracies.append(np.mean(batch_accuracies))

        # and get average losses across the batches
        losses[epoch_i, 0] = float(np.mean(batch_loss))

        # test accuracy
        net.eval()
        X, y = next(iter(val_loader))  # extract X,y from test dataloader
        with torch.no_grad():  # deactivates autograd
            y_hat = net(X)

        # compare the following really long line of code to the training accuracy lines
        val_accuracies.append(100 * torch.mean((torch.argmax(y_hat, dim=1) == y).float()))
        val_loss = loss_func(y_hat, y)
        losses[epoch_i, 1] = val_loss.item()
        print(f"Epoch {epoch_i + 1}/{num_epochs}, Train Loss: {losses[epoch_i, 0]:.4f}, Val Loss: {losses[epoch_i, 1]:.4f}")
    # end epochs

    # function output
    return train_accuracies, val_accuracies, losses, net

In [ ]:
train_accuracies, val_accuracies, losses, net = train_cifar_10(num_epochs=15, learning_rate=0.001)

In [ ]:
# Calculate test accuracy
X_test, y_test = next(iter(test_loader))
with torch.no_grad():
    y_pred = net(X_test)
test_accuracy = 100 * (y_pred.argmax(dim=1) == y_test).float().mean().item()

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].plot(losses, label="Train Loss")
ax[0].set_xlabel("Epochs")
ax[0].set_ylabel("Loss")
ax[0].set_title("Training Loss")
ax[0].legend(["Train Loss", "Validation Loss"])

ax[1].plot(train_accuracies, label="Train Accuracy")
ax[1].plot(val_accuracies, label="Validation Accuracy")
ax[1].axhline(
    test_accuracy,
    color="red",
    linestyle="--",
    label=f"Test Accuracy: {test_accuracy:.2f}%",
)
ax[1].set_xlabel("Epochs")
ax[1].set_ylabel("Accuracy (%)")
ax[1].set_title(f"Training and Validation Accuracy: {val_accuracies[-1]:.2f}%")
ax[1].set_ylim(0, 100)
ax[1].legend()